# Medtext GPT2 Fine tuning

## Dataset Preparation

This is the foundation of the fine-tuning pipeline. Proper data management and preparation are critical for the model’s performance.

1. **Data Collection and Curation:** Gather a high-quality dataset relevant to the target domain, ensuring it covers diverse scenarios.
2. **Data Cleaning and QA:** Remove inconsistencies, irrelevant data, and errors to ensure the dataset is accurate and reliable.
3. **Data Augmentation:** Enrich the dataset by generating synthetic samples, applying paraphrasing, or augmenting low-resource data.
4. **Data Splitting:** Divide the data into training, validation, and test sets to facilitate proper evaluation.

In [4]:
from src.dataset_preparation import prepare_dataset

prepare_dataset()

FileExistsError: data/processed/medtext_v1 already exists

### Load Dataset

In [5]:
from datasets import load_from_disk

DATASET_PATH = "data/processed/medtext_v1"


def load_medtext():
    dataset = load_from_disk(DATASET_PATH)
    return dataset["train"], dataset["validation"], dataset["test"]


train_data, val_data, test_data = load_medtext()

## Tokenization

1. First we need to tokenize all data using the same tokenizer used in GPT2
2. Then we need to create the Dataloader using pad since the sequences have diferent lenghts

In [6]:
import importlib
import src.tokenizer as tokenizer_module

importlib.reload(tokenizer_module)
from src.tokenizer import tokenize_dataset, collate_fn
from torch.utils.data import DataLoader

# [Step 1] First we need to tokenize all data
train_tokenized = tokenize_dataset(train_data)
val_tokenized = tokenize_dataset(val_data)
test_tokenized = tokenize_dataset(test_data)

# [Step 2] Now we need to create the DataLoaders for training, validating and testing

train_loader = DataLoader(
    train_tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

## Load GPT Model

In [ ]:
from torch.optim import AdamW
from src.gpt import load_model
from src.evaluation import evaluate

# [Step 1 ] Load the model (in this case gpt2)
# with the pretrained weights from the hugging face
model, device = load_model("openai-community/gpt2")

# [Step 2] Create the optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# [Step 3] evaluate the model before the Fine tuning
baseline_loss = evaluate(model, val_loader, device)
print(f"Baseline validation loss: {baseline_loss:.4f}")

W1101 19:49:03.853000 58316 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7480.08it/s]


Device: mps
Parameters: 124,439,808


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline validation loss: 2.8881


## Training Loop
Now that we already have the baseline loss for this model we need to create the training loop to fine tune our model

In [10]:
from pathlib import Path
import torch

best_val_loss = float("inf")
EPOCHS = 2

for epoch in range(EPOCHS):

    # [Step 1] Set model to training mode
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for idx, batch in enumerate(train_loader):
        if idx >= 5:
            break

        # [Step 2] Move tensors to device
        batch = {key: value.to(device) for key, value in batch.items()}

        # [Step 3] Reset gradients
        optimizer.zero_grad(set_to_none=True)

        # [Step 4] Forward pass
        outputs = model(**batch)
        loss = outputs.loss

        # [Step 5] Backpropagation
        loss.backward()

        # [Step 6] Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # [Step 7] Update model weights
        optimizer.step()

        # Track training loss
        num_tokens = (batch["labels"][:, 1:] != -100).sum().item()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

    train_loss = total_loss / total_tokens

    # [Step 8] Validation
    val_loss = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # [Step 9] Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        output_dir = Path("checkpoints/gpt2-medtext-best")
        output_dir.mkdir(parents=True, exist_ok=True)

        model.save_pretrained(output_dir)

KeyboardInterrupt: 